In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\karth\AppData\Local\Temp\ipykernel_23180\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [4]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
        
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("C:\Projects\Ragbase\data\pdf")

Found 6 PDF files in C:\Projects\Ragbase\data\pdf
Processing C:\Projects\Ragbase\data\pdf\computer_networks_basics.pdf...
Processing C:\Projects\Ragbase\data\pdf\database_management_systems.pdf...
Processing C:\Projects\Ragbase\data\pdf\IEFT-Module_1.pdf...
Processing C:\Projects\Ragbase\data\pdf\IEFT_Module-2.pdf...
Processing C:\Projects\Ragbase\data\pdf\machine_learning_intro.pdf...
Processing C:\Projects\Ragbase\data\pdf\operating_systems_overview.pdf...
Total documents loaded: 39


In [5]:
all_pdf_documents


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-01T10:00:54+00:00', 'source': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'file_path': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-01T10:00:54+00:00', 'trapped': '', 'modDate': "D:20260601100054+00'00'", 'creationDate': "D:20260601100054+00'00'", 'page': 0}, page_content='Introduction to Computer Networks\nComputer networks allow devices to communicate and share resources. The internet is the largest\nexample of a computer network.\nNetworks are commonly classified as LAN, MAN, and WAN depending on their size and coverage area.\nProtocols such as TCP/IP define the rules for communication between devices.\nRouters, switches, and firewalls are important networking dev

In [24]:


def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len)
    split_docs= text_splitter.split_documents(documents)
    print(f'split {len(documents)} documents into {len(split_docs)} chunks')

    #example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"contents: {split_docs[0].page_content[:200]}")
        print(f"...\nmetadata: {split_docs[0].metadata}")

    return split_docs



In [25]:
chunks=split_documents(all_pdf_documents)
chunks

split 39 documents into 88 chunks

Example chunk:
contents: Introduction to Computer Networks
Computer networks allow devices to communicate and share resources. The internet is the largest
example of a computer network.
Networks are commonly classified as LAN
...
metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-01T10:00:54+00:00', 'source': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'file_path': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-01T10:00:54+00:00', 'trapped': '', 'modDate': "D:20260601100054+00'00'", 'creationDate': "D:20260601100054+00'00'", 'page': 0}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-01T10:00:54+00:00', 'source': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'file_path': 'C:\\Projects\\Ragbase\\data\\pdf\\computer_networks_basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-01T10:00:54+00:00', 'trapped': '', 'modDate': "D:20260601100054+00'00'", 'creationDate': "D:20260601100054+00'00'", 'page': 0}, page_content='Introduction to Computer Networks\nComputer networks allow devices to communicate and share resources. The internet is the largest\nexample of a computer network.\nNetworks are commonly classified as LAN, MAN, and WAN depending on their size and coverage area.\nProtocols such as TCP/IP define the rules for communication between devices.\nRouters, switches, and firewalls are important networking dev

In [26]:
#EMBEDDING AND VECTOR STORE

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
class EmbeddingManager:
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model= None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading Emmeding Model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded succesfully, embedding dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model{self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts:List[str]) -> np.ndarray:

        if not self.model:
            raise ValueError("model not found")
        
        print(f"Generating embedding for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generating embedding with shape: {embeddings.shape}")
        return embeddings
    

In [29]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading Emmeding Model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model loaded succesfully, embedding dimensions: 384


C:\Users\karth\AppData\Local\Temp\ipykernel_23180\1644642136.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded succesfully, embedding dimensions: {self.model.get_sentence_embedding_dimension()}")


In [31]:
class VectorStore:
    def __init__(self, collection_name:str="pdf_documents", persist_directory:str="./vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(Path(self.persist_directory))
            
            self.collection=self.client.get_or_create_collection(name=self.collection_name)
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents and embeddings must match")
        print(f"Adding {len(documents)} documents to vector store...")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            metadatas.append(metadata)


            documents_text.append(doc.page_content)


            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(ids=ids, metadatas=metadatas, documents=documents_text, embeddings=embeddings_list)
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

        



In [33]:
vectorstore=VectorStore()
vectorstore


Vector store initialized with collection: pdf_documents
existing documents in collection: 282


In [34]:
texts=[doc.page_content for doc in chunks]
texts

['Introduction to Computer Networks\nComputer networks allow devices to communicate and share resources. The internet is the largest\nexample of a computer network.\nNetworks are commonly classified as LAN, MAN, and WAN depending on their size and coverage area.\nProtocols such as TCP/IP define the rules for communication between devices.\nRouters, switches, and firewalls are important networking devices used in modern systems.\nCybersecurity is an important aspect of networking to protect data from attacks.\nTopic\nImportance\nConcept Understanding\nBuilds foundational CS knowledge\nPractical Usage\nUseful for projects and applications\nCareer Relevance\nImportant for technical interviews',
 'Database Management Systems\nA Database Management System (DBMS) is software used to store and manage data efficiently.\nRelational databases organize data into tables with rows and columns.\nSQL is the standard language used to query and manage relational databases.\nNormalization reduces redund

In [35]:
embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embedding for 88 texts...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generating embedding with shape: (88, 384)
Adding 88 documents to vector store...
Documents added successfully. Total documents in collection: 370


In [36]:
print(len(chunks))
print(len(texts))
print(len(embeddings))

88
88
88


In [38]:
class RAGRetriever:
    def __init__(self, vector_store:VectorStore, embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    
    def retrieve(self, query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(query_embeddings=[query_embedding], n_results=top_k)
            retrieved_docs=[]
            if results["documents"] and results["documents"][0]:
                documents=results["documents"][0]
                metadatas=results["metadatas"][0]
                distances=results["distances"][0]
                ids=results["ids"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(documents, metadatas, documents, distances)):
                similarity_score= 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "content": document,
                        "similarity Score": similarity_score,
                        "distance": distance,
                        "rank": i + 1
                    })
            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise

In [18]:
rag_retriever=RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [39]:
rag_retriever.retrieve("machine learning is the subset of what?")

Retrieving documents for query: 'machine learning is the subset of what?'
Top K: 5, Score Threshold: 0.0
Generating embedding for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embedding with shape: (1, 384)
Retrieved 4 documents


[{'id': 'Basics of Machine Learning\nMachine Learning is a field of Artificial Intelligence that allows systems to learn from data.\nSupervised learning uses labeled datasets to train models for prediction tasks.\nUnsupervised learning finds hidden patterns in data without labels.',
  'metadata': {'keywords': '',
   'content_length': 269,
   'total_pages': 1,
   'page': 0,
   'subject': '(unspecified)',
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'trapped': '',
   'creator': '(unspecified)',
   'creationdate': '2026-06-01T10:00:54+00:00',
   'moddate': '2026-06-01T10:00:54+00:00',
   'file_path': 'C:\\Projects\\Ragbase\\data\\pdf\\machine_learning_intro.pdf',
   'author': '(anonymous)',
   'source': 'C:\\Projects\\Ragbase\\data\\pdf\\machine_learning_intro.pdf',
   'format': 'PDF 1.4',
   'modDate': "D:20260601100054+00'00'",
   'creationDate': "D:20260601100054+00'00'",
   'doc_index': 252,
   'title': '(anonymous)'},
  'content': 'Basics of Machine Learning\nMachin

In [41]:
#simple rag pipeline with groq
from urllib import response

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant")

def rag_simple(query, retriever, llm, top_k=5):
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        print("No relevant documents found, returning LLM response without context.")
        
    prompt=f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    response=llm.invoke(prompt.format(context=context, query=query))
    return response.content


In [42]:
answer=rag_simple("what is machine learning?", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'what is machine learning?'
Top K: 5, Score Threshold: 0.0
Generating embedding for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embedding with shape: (1, 384)
Retrieved 5 documents
Answer: Machine Learning is a field of Artificial Intelligence that allows systems to learn from data.


In [49]:
def rag_advanced(query, retriever, llm, top_k=5,min_score=0.2,return_context=False):
    results=retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        print("No relevant documents found, returning LLM response without context.")

    context="\n\n".join([doc["content"] for doc in results]) if results else ""
    sources=[{'source': doc['metadata'].get('source', 'unknown'), 'rank': doc['rank'], 'page': doc['metadata'].get('page', 'unknown')} for doc in results] if results else []
    confidence=max([doc['similarity Score'] for doc in results]) if results else 0.0
    prompt=f"use the following context to answer the question concisely. \n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    response=llm.invoke(prompt.format(query=query))

    output={
        "Query": query,
        "answer": response.content,
        "context": context,
        "sources": sources,
        "confidence": confidence
        }
    if return_context:
        output["context"]=context
    return output


In [50]:
result=rag_advanced("Cobb – Douglas Production function", rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)
print("Answer:", result["answer"])
print("Context:", result["context"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])


Retrieving documents for query: 'Cobb – Douglas Production function'
Top K: 5, Score Threshold: 0.2
Generating embedding for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embedding with shape: (1, 384)
Retrieved 5 documents
Answer: The Cobb-Douglas Production function is used to represent the technological relationship between the amounts of two inputs (Capital and Labour), and the amount of output that can be produced by those inputs.
Context:  The Cobb-Douglas Production function is a homogeneous production function of degree 
one. That is α+β=1. 
*If α+β=1, it is the case of constant returns to scale. 
*If α+β>1, it is the case of increasing returns to scale. 
*If α+β<1, it is the case of decreasing returns to scale. 
 MPL= α× (Q/L)

can be produced by those inputs. 
 This function is written as; Q=ALα Kβ 
Q- Total output, A-Total factor productivity, L- Labour, K- Capital, 
α and β are the output elasticities of labour and capital. 
 The Cobb-Douglas Production function is a homogeneous production function of degree 
one. That is α+β=1. 
*If α+β=1, it is the case of constant returns to scale. 
*If α+β>1, it is the case of increasing r